# Vertex Coloring
Abbiamo un grafo **non orientato**, il modo di rappresentarlo è il seguente:
$$
G = \left(V,E\right)
$$ 
dove $V$ è l'insieme dei vertici e $E$ è l'insieme degli archi del grafo.
# Obiettivo
Vogliamo colorare ogni vertice del grafo 
1. con meno colori possibili 
2. senza che due vertici collegati abbiano lo stesso colore
$V=\{1,2,3,4,5\}$ e  $E=\{(1,2),(2,4),(3,4),(3,5),(4,5)\}$
```mermaid
graph LR
    n1(("1")) --- n2(("2"))
    n2 --- n4(("4"))
    n3(("3")) --- n4
    n3 --- n5(("5"))
    n4 --- n5

```
> Esempio di soluzione **non ottima**: ogni vertice ha un colore diverso.
```mermaid
graph LR
    n1(("1")) --- n2(("2"))
    n2 --- n4(("4"))
    n3(("3")) --- n4
    n3 --- n5(("5"))
    n4 --- n5

    classDef c1 fill:#e74c3c,stroke:#333,stroke-width:2px,color:#fff;
    classDef c2 fill:#3498db,stroke:#333,stroke-width:2px,color:#fff;
    classDef c3 fill:#2ecc71,stroke:#333,stroke-width:2px,color:#fff;
    classDef c4 fill:#f1c40f,stroke:#333,stroke-width:2px,color:#000;
    classDef c5 fill:#9b59b6,stroke:#333,stroke-width:2px,color:#fff;

    class n1 c1;
    class n2 c2;
    class n3 c3;
    class n4 c4;
    class n5 c5;

```

# Modellazione ILP
Vogliamo modellare il problem del vertex coloring dato un insieme di vertici, un insieme di colori e un grafo $G=(V,E)$
Modello ILP:

$\displaystyle \min \sum_{j \in colori} y_j\quad$ ovvero minimizzare il numero di colori utilizzati

$ \displaystyle\sum_{j \in colori} x_{ij} = 1 \quad i \in vertici \quad$ ovvero ogni vertice deve essere colorato con un solo colore

$  x_{ij} + x_{hj}\le  y_j \quad j \in colori, ~(i,h) \in lati \quad$ ovvero se due vertici sono adiacenti non possono essere colorati con lo stesso colore

$ x_{ij} \in \{0,1\} \quad i \in vertici,~j \in colori$ ovvero $x_{ij}\quad$ è una variabile binaria che indica se il vertice $i$ è colorato con il colore $j$ o meno

$ y_j \in \{0,1\} \quad j \in colori\quad$ ovvero $y_j$ è una variabile binaria che indica se il colore $j$ è utilizzato o meno

# Lettura dati da file
Leggeremo i dati del grafo da un file, la struttura del file che conterrà
```
5 5
1 2
2 4
3 4
3 5
4 5
```
che va interpretato così 
1. La prima riga `5 5`
   - `5` = numero di vertici `n`
   - `5` = numero di lati `m`
2. Ogni riga successiva contiene una coppia di vertici collegati da un lato:
    - `1 2` significa lato tra `1` e `2`
    - `2 4` significa lato tra `2` e `4`
    - `3 4` significa lato tra `3` e `4`
    - etc



In [1]:
input=open("05.0 Dati.txt", "r")
riga=input.readline().split()
n=int(riga[0])
m=int(riga[1])
print('|V|=n=', n)
print('|E|=m=', m)
vertici=[]
for i in range(1,n+1):
    vertici.append(i)
print("Vertici =",vertici)
archi=[]
for i in range(m):
    riga=input.readline().split()
    u=int(riga[0])
    v=int(riga[1])
    archi.append((u, v))#Uso una tupla perché non è un grafo orientato
print("Archi =",archi)

|V|=n= 5
|E|=m= 5
Vertici = [1, 2, 3, 4, 5]
Archi = [(1, 2), (2, 4), (3, 4), (3, 5), (4, 5)]


In [2]:
import pyomo.environ as pyo

model = pyo.ConcreteModel(name='Vertex Coloring')

colori = ["color" + str(i) for i in range(1, len(vertici) + 1)]
print(colori)
model.y = pyo.Var(colori, within=pyo.Binary)
model.x = pyo.Var(vertici, colori, within=pyo.Binary)

def funzione_obiettivo(m):
    return sum(m.y[j] for j in colori)
model.obiettivo = pyo.Objective(rule=funzione_obiettivo, sense=pyo.minimize)

# Un vertice deve essere colorato con esattamente un colore
def vincolo_assegnamento_unico(model, vertice):
    return sum(model.x[vertice, colore] for colore in colori) == 1
model.vincolo1 = pyo.Constraint(vertici, rule=vincolo_assegnamento_unico)

# Un colore può essere assegnato a un vertice solo se è attivo
def vincolo_attivazione_colore(model, vertice, colore):
    return model.x[vertice, colore] <= model.y[colore]
model.vincolo2 = pyo.Constraint(vertici, colori, rule=vincolo_attivazione_colore)

# Evitare che due vertici adiacenti abbiano lo stesso colore
def vincolo_non_conflitto(model, vertice1, vertice2, colore):
    if (vertice1, vertice2) in archi or (vertice2, vertice1) in archi:
        return model.x[vertice1, colore] + model.x[vertice2, colore] <= 1
    return pyo.Constraint.Skip
model.vincolo3 = pyo.Constraint(vertici, vertici, colori, rule=vincolo_non_conflitto)

ottimizzatore=pyo.SolverFactory('glpk')
risultato=ottimizzatore.solve(model, tee=False)

model.display()

['color1', 'color2', 'color3', 'color4', 'color5']
Model Vertex Coloring

  Variables:
    y : Size=5, Index={color1, color2, color3, color4, color5}
        Key    : Lower : Value : Upper : Fixed : Stale : Domain
        color1 :     0 :   0.0 :     1 : False : False : Binary
        color2 :     0 :   0.0 :     1 : False : False : Binary
        color3 :     0 :   1.0 :     1 : False : False : Binary
        color4 :     0 :   1.0 :     1 : False : False : Binary
        color5 :     0 :   1.0 :     1 : False : False : Binary
    x : Size=25, Index={1, 2, 3, 4, 5}*{color1, color2, color3, color4, color5}
        Key           : Lower : Value : Upper : Fixed : Stale : Domain
        (1, 'color1') :     0 :   0.0 :     1 : False : False : Binary
        (1, 'color2') :     0 :   0.0 :     1 : False : False : Binary
        (1, 'color3') :     0 :   1.0 :     1 : False : False : Binary
        (1, 'color4') :     0 :   0.0 :     1 : False : False : Binary
        (1, 'color5') :     0 :

## Versione ottimizzata per file grande

In [3]:
import pyomo.environ as pyo

TEMPO_MASSIMO_SECONDS = 60
MIP_GAP = 0.05

# lettura dati
with open("05.1 Dati.txt", "r") as f:
    prima_riga = f.readline().split()
    n = int(prima_riga[0])
    m = int(prima_riga[1])

    archi = []
    for _ in range(m):
        u, v = map(int, f.readline().split())
        archi.append((u, v))

vertici = list(range(1, n + 1))

# Trova subito una colorazione ammissibile. Il numero di colori trovato diventa
# un limite superiore: evita di creare n colori quando non servono.
adiacenti = {v: set() for v in vertici}
for u, v in archi:
    adiacenti[u].add(v)
    adiacenti[v].add(u)

colorazione_greedy = {}
for vertice in sorted(vertici, key=lambda v: len(adiacenti[v]), reverse=True):
    colori_vietati = {colorazione_greedy[v] for v in adiacenti[vertice] if v in colorazione_greedy}
    colore = 1
    while colore in colori_vietati:
        colore += 1
    colorazione_greedy[vertice] = colore

colori = list(range(1, max(colorazione_greedy.values()) + 1))
print(f"Colori iniziali greedy: {len(colori)}")

model = pyo.ConcreteModel(name="Vertex Coloring")

model.y = pyo.Var(colori, within=pyo.Binary)
model.x = pyo.Var(vertici, colori, within=pyo.Binary)

for colore in colori:
    model.y[colore].value = 1 if colore <= len(colori) else 0
for vertice in vertici:
    for colore in colori:
        model.x[vertice, colore].value = int(colorazione_greedy[vertice] == colore)

def funzione_obiettivo(model):
    return sum(model.y[colore] for colore in colori)

model.obiettivo = pyo.Objective(rule=funzione_obiettivo, sense=pyo.minimize)

def vincolo_assegnamento_unico(model, vertice):
    return sum(model.x[vertice, colore] for colore in colori) == 1

model.vincolo1 = pyo.Constraint(vertici, rule=vincolo_assegnamento_unico)

def vincolo_attivazione_colore(model, vertice, colore):
    return model.x[vertice, colore] <= model.y[colore]

model.vincolo2 = pyo.Constraint(vertici, colori, rule=vincolo_attivazione_colore)

def vincolo_non_conflitto(model, u, v, colore):
    return model.x[u, colore] + model.x[v, colore] <= 1

model.vincolo3 = pyo.Constraint(archi, colori, rule=vincolo_non_conflitto)

def vincolo_ordine_colori(model, colore):
    if colore == colori[-1]:
        return pyo.Constraint.Skip
    return model.y[colore] >= model.y[colore + 1]

model.vincolo4 = pyo.Constraint(colori, rule=vincolo_ordine_colori)

solver = pyo.SolverFactory("glpk")
solver.options["tmlim"] = TEMPO_MASSIMO_SECONDS
solver.options["mipgap"] = MIP_GAP
results = solver.solve(model, tee=False)

print("Stato solver:", results.solver.status)
print("Terminazione:", results.solver.termination_condition)

colori_usati = [colore for colore in colori if pyo.value(model.y[colore]) > 0.5]
soluzione = {
    vertice: next(colore for colore in colori if pyo.value(model.x[vertice, colore]) > 0.5)
    for vertice in vertici
}

print("Colori usati:", len(colori_usati))
print("Assegnamento vertice -> colore:")
print(soluzione)


Colori iniziali greedy: 5
Stato solver: ok
Terminazione: feasible
Colori usati: 5
Assegnamento vertice -> colore:
{1: 1, 2: 1, 3: 1, 4: 1, 5: 5, 6: 1, 7: 1, 8: 3, 9: 1, 10: 1, 11: 1, 12: 1, 13: 1, 14: 3, 15: 1, 16: 1, 17: 1, 18: 2, 19: 5, 20: 1, 21: 3, 22: 2, 23: 5, 24: 3, 25: 2, 26: 1, 27: 3, 28: 1, 29: 2, 30: 3, 31: 1, 32: 1, 33: 2, 34: 1, 35: 3, 36: 2, 37: 4, 38: 1, 39: 1, 40: 3, 41: 5, 42: 1, 43: 1, 44: 3, 45: 3, 46: 4, 47: 1, 48: 1, 49: 5, 50: 1, 51: 2, 52: 1, 53: 5, 54: 4, 55: 5, 56: 2, 57: 3, 58: 5, 59: 1, 60: 5, 61: 2, 62: 3, 63: 2, 64: 5, 65: 2, 66: 2, 67: 5, 68: 3, 69: 4, 70: 1, 71: 3, 72: 3, 73: 1, 74: 1, 75: 3, 76: 3, 77: 4, 78: 5, 79: 5, 80: 4, 81: 5, 82: 3, 83: 5, 84: 3, 85: 3, 86: 2, 87: 1, 88: 5, 89: 4, 90: 2, 91: 3, 92: 3, 93: 1, 94: 2, 95: 5, 96: 5, 97: 3, 98: 5, 99: 5, 100: 3, 101: 3, 102: 2, 103: 3, 104: 3, 105: 5, 106: 5, 107: 3, 108: 5, 109: 5, 110: 5, 111: 5, 112: 5, 113: 1, 114: 1, 115: 2, 116: 3, 117: 5, 118: 1, 119: 1, 120: 3, 121: 4, 122: 5, 123: 3, 124: 1, 1